In [1]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

2025-04-23 23:40:00.063926: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-23 23:40:00.100061: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745440800.142683    6899 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745440800.153856    6899 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-23 23:40:00.197613: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
FOLDER_PATH = "../img/v"
FRAME_COUNT = 16
IMG_SIZE = 64
CLASS_COUNT = 3
class_names = ["0", "1", "2"]

In [3]:
def load_videos(folder, frame_count=FRAME_COUNT, img_size=IMG_SIZE):
    videos = []
    labels = []

    for video_folder in os.listdir(folder):
        video_path = os.path.join(folder, video_folder)
        if os.path.isdir(video_path):
            try:
                _, label = video_folder.split("_")
                label = int(label)
            except ValueError:
                print(f"[SKIP] invalid dir name: {video_folder}")
                continue

            frames = []
            frame_files = sorted(os.listdir(video_path))[:frame_count]
            print(f"[INFO] {video_folder} -> {len(frame_files)} frame")

            for filename in frame_files:
                img_path = os.path.join(video_path, filename)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    img = cv2.resize(img, (img_size, img_size))
                    img = img.flatten()
                    frames.append(img)
                else:
                    print(f"[WARN] frame can not be read!: {img_path}")

            if len(frames) == frame_count:
                videos.append(frames) 
                labels.append(label)
            else:
                print(f"[SKIP] {video_folder} -> not enough frame ({len(frames)}/{frame_count})")

    print(f"[SUMMARY] Sum of videos: {len(videos)}")
    return np.array(videos), np.array(labels)

In [4]:
X, y = load_videos(FOLDER_PATH, FRAME_COUNT, IMG_SIZE)
y = to_categorical(y, num_classes=CLASS_COUNT)

[INFO] 54_0 -> 16 frame
[INFO] 51_0 -> 16 frame
[INFO] 120_0 -> 16 frame
[INFO] 40_1 -> 16 frame
[INFO] 38_1 -> 16 frame
[INFO] 22_0 -> 16 frame
[INFO] 60_0 -> 16 frame
[INFO] 64_1 -> 16 frame
[INFO] 96_2 -> 16 frame
[INFO] 137_1 -> 16 frame
[INFO] 36_2 -> 16 frame
[INFO] 101_1 -> 16 frame
[INFO] 139_1 -> 16 frame
[INFO] 103_2 -> 16 frame
[INFO] 18_1 -> 16 frame
[INFO] 129_1 -> 16 frame
[INFO] 21_0 -> 16 frame
[INFO] 34_2 -> 16 frame
[INFO] 43_1 -> 16 frame
[INFO] 83_2 -> 16 frame
[INFO] 109_2 -> 16 frame
[INFO] 32_2 -> 16 frame
[INFO] 39_1 -> 16 frame
[INFO] 119_1 -> 16 frame
[INFO] 85_1 -> 16 frame
[INFO] 58_2 -> 16 frame
[INFO] 25_1 -> 16 frame
[INFO] 24_1 -> 16 frame
[INFO] 100_2 -> 16 frame
[INFO] 42_1 -> 16 frame
[INFO] 89_1 -> 16 frame
[INFO] 66_0 -> 16 frame
[INFO] 135_2 -> 16 frame
[INFO] 35_1 -> 16 frame
[INFO] 88_1 -> 16 frame
[INFO] 112_2 -> 16 frame
[INFO] 23_2 -> 16 frame
[INFO] 27_0 -> 16 frame
[INFO] 16_2 -> 16 frame
[INFO] 94_2 -> 16 frame
[INFO] 62_2 -> 16 frame
[INFO

In [5]:
print("Number of videos:", len(X))
print("Shape of data:", X.shape) 

Number of videos: 141
Shape of data: (141, 16, 4096)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [9]:
model = Sequential([
    LSTM(128, input_shape=(FRAME_COUNT, IMG_SIZE * IMG_SIZE), return_sequences=False),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.1),
    Dense(CLASS_COUNT, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

model.fit(X_train, y_train, epochs=10, batch_size=4, validation_split=0.1)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 128)            │     2,163,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,171,651 (8.28 MB)

 Trainable params: 2,171,651 (8.28 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 10s 201ms/step - accuracy: 0.3139 - loss: 1.2733 - val_accuracy: 0.5833 - val_loss: 0.9277
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 172ms/step - accuracy: 0.4157 - loss: 1.0617 - val_accuracy: 0.3333 - val_loss: 1.1751
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 174ms/step - accuracy: 0.3376 - loss: 1.1823 - val_accuracy: 0.5833 - val_loss: 0.9547
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 175ms/step - accuracy: 0.3571 - loss: 1.1088 - val_accuracy: 0.5833 - val_loss: 0.9543
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 172ms/step - accuracy: 0.4416 - loss: 1.0693 - val_accuracy: 0.5833 - val_loss: 0.9685
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 179ms/step - accuracy: 0.5001 - loss: 1.0518 - val_accuracy: 0.5833 - val_loss: 0.9837
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 182ms/step - accuracy: 0.5093 - loss: 1.0917 - val_accuracy: 0.5833 - val_loss: 0.9766
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 175ms/step - accuracy: 0.4629 - loss: 1.0582 - val_accuracy: 0

In [8]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"LSTM Model Test Accuracy: {test_acc:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 934ms/step - accuracy: 0.3448 - loss: 1.1217
LSTM Model Test Accuracy: 0.34
